# Doc-to-LoRA: Knowledge Injection into MLP Layers for Nemotron-3-Nano-30B

## Overview

This notebook implements a **2-phase knowledge injection + task SFT** pipeline inspired by [Doc-to-LoRA (Sakana AI, 2025)](https://arxiv.org/abs/2602.15902). The key insight is that transformer MLP/FFN layers act as key-value memory stores for factual knowledge ([Dai et al., 2022](https://arxiv.org/abs/2104.08696)), and LoRA adapters targeting these layers can embed lookup-table knowledge directly into model weights.

### Why Doc-to-LoRA for this competition?

The **Text Encryption** puzzle type requires the model to know a fixed 77-word dictionary to decode substitution ciphers. A naive approach (listing all 77 words in the Chain-of-Thought) wastes ~365 tokens per problem and actually **hurts** accuracy (35% -> 22%). Instead, we embed this dictionary knowledge into the LoRA weights via pattern-matching QA training, freeing the CoT to focus on cipher solving.

### Pipeline

```
Step 1: Load pre-built training data (knowledge QA + 6-type CoT)
        - Pattern Match QA: "Which Wonderland word matches ?oo?" -> "book, door"
        - Dictionary Recall: "List all 3-letter Wonderland words" -> "cat, key, map, the"
        - Encryption tasks with SHORT CoT (no 77-word listing)
        
Step 2: Knowledge Injection SFT (Phase 1)
        - LoRA (rank=32, alpha=64) on in/out/up/down_proj
        - Higher LR (1e-4) to deeply embed dictionary patterns
        - Data: knowledge QA + encryption short CoT (~3,000 samples)
        
Step 3: Task SFT (Phase 2)
        - Continue from Phase 1 adapter
        - Lower LR (5e-5) to preserve knowledge while learning task format
        - Data: all 6 puzzle types (~3,400 samples)
        
Step 4: Create submission.zip
```

### Results (local eval, 500 samples)

| Puzzle Type | Base Model | This Method |
|---|:---:|:---:|
| Text Encryption | 34.2% | **74.7%** |
| Gravitational Constant | 98.8% | **100%** |
| Unit Conversion | 74.1% | **100%** |
| Numeral Conversion | 100% | **100%** |
| Bit Manipulation | 9.6% | 13.8% |
| Equation Transformation | 8.6% | 12.3% |
| **Overall Accuracy** | **52.8%** | **65.4%** |

### References

- **Doc-to-LoRA**: Sakana AI (2025). *Instant Document-Grounded LLM Reasoning via LoRA*. [arXiv:2602.15902](https://arxiv.org/abs/2602.15902) | [GitHub](https://github.com/SakanaAI/doc-to-lora)
- **Knowledge Neurons**: Dai et al. (2022). *Knowledge Neurons in Pretrained Transformers*. [arXiv:2104.08696](https://arxiv.org/abs/2104.08696)
- **MEMIT**: Meng et al. (2023). *Mass-Editing Memory in a Transformer*. [arXiv:2210.07229](https://arxiv.org/abs/2210.07229)
- **ProDiaL**: (2025). *Parameter Efficient Mamba Tuning via Projector-targeted Diagonal-centric Linear Transformation*. [arXiv:2411.15224](https://arxiv.org/abs/2411.15224) (validates in_proj/out_proj targeting for Mamba layers)

## Environment Setup

Kaggle Blackwell GPU (RTX Pro 6000) requires special patches for Triton and RMSNorm.

In [1]:
import os, sys, subprocess, shutil, importlib, glob, types

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- CRITICAL: Pre-inject dummy modules for mamba3/cutlass ---
# The utility script's mamba_ssm.__init__.py does:
#   from mamba_ssm.modules.mamba3 import Mamba3
# which imports cutlass (C++ bindings not on Kaggle).
# Nemotron-H uses Mamba2 only, so we pre-inject dummies into sys.modules
# BEFORE mamba_ssm gets imported. When Python finds a key in sys.modules,
# it returns it immediately without going through finders/loaders.

def _make_dummy(name, attrs=None):
    """Create a dummy module and register in sys.modules."""
    mod = types.ModuleType(name)
    mod.__path__ = []  # make it look like a package
    mod.__package__ = name
    mod.__file__ = f"<dummy {name}>"
    if attrs:
        for k, v in attrs.items():
            setattr(mod, k, v)
    sys.modules[name] = mod
    return mod

# Pre-inject all cutlass-related modules
for modname in [
    "cutlass", "cutlass.cute", "cutlass.cutlass_dsl",
    "cutlass._mlir", "cutlass._mlir.dialects",
    "cutlass._mlir.dialects.llvm", "cutlass._mlir.dialects.nvvm",
    "cutlass._mlir.dialects.vector",
]:
    _make_dummy(modname, {"Int32": None, "Float32": None, "Float16": None,
                          "BFloat16": None, "Boolean": None, "const_expr": lambda x: x,
                          "T": None, "dsl_user_op": None})

# Pre-inject cuda.bindings stubs (mamba3_step_fn imports cuda.bindings.driver)
for modname in ["cuda", "cuda.bindings", "cuda.bindings.driver"]:
    if modname not in sys.modules:
        _make_dummy(modname)

# Pre-inject mamba3 module stubs
for modname in [
    "mamba_ssm.ops.cute",
    "mamba_ssm.ops.cute.mamba3",
    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
    "mamba_ssm.modules.mamba3",
    "mamba_ssm.ops.triton.mamba3",
    "mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step",
]:
    _make_dummy(modname, {"Mamba3": None, "mamba3_step_fn": None,
                          "apply_rotary_qk_inference_fwd": None})

# Clear any partially-loaded mamba_ssm to force clean re-import
for key in list(sys.modules.keys()):
    if key.startswith("mamba_ssm") and key not in [
        "mamba_ssm.ops.cute", "mamba_ssm.ops.cute.mamba3",
        "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
        "mamba_ssm.modules.mamba3",
        "mamba_ssm.ops.triton.mamba3",
        "mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step",
    ]:
        del sys.modules[key]

print("Pre-injected cutlass/mamba3 dummy modules into sys.modules")

# --- Resolve utility script for Blackwell GPU ---
UTILITY_SCRIPT_DIR = "/kaggle/input/nvidia-utility-script"
if os.path.exists(UTILITY_SCRIPT_DIR):
    pth_files = glob.glob(os.path.join(UTILITY_SCRIPT_DIR, "**/*.pth"), recursive=True)
    for pth in pth_files:
        with open(pth) as f:
            for line in f:
                p = line.strip()
                if p and os.path.isdir(p) and p not in sys.path:
                    sys.path.insert(0, p)
    print(f"Loaded {len(pth_files)} utility .pth files")

# --- Install offline packages (datasets, trl) ---
OFFLINE_DIR = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
TARGET_DIR = "/kaggle/tmp/lib"
if os.path.exists(OFFLINE_DIR):
    os.makedirs(TARGET_DIR, exist_ok=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", OFFLINE_DIR,
        "--target", TARGET_DIR, "datasets", "trl"
    ])
    sys.path.insert(0, TARGET_DIR)
    print(f"Installed offline packages to {TARGET_DIR}")

print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Pre-injected cutlass/mamba3 dummy modules into sys.modules
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.12.0.dev20260324+cu128, CUDA: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [2]:
# pip show trl
!pip install --no-index --find-links=/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/ trl

Looking in links: /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/trl-0.29.1-py3-none-any.whl


In [3]:
# --- Blackwell GPU patches (continued) ---

# Patch 1: Triton ptxas for Blackwell
try:
    import triton
    ptxas_src = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    triton_dir = os.path.dirname(triton.__file__)
    dst_dir = os.path.join(triton_dir, "third_party", "cuda", "bin")
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, "ptxas")
    if not os.path.exists(dst) and os.path.exists(ptxas_src):
        shutil.copy2(ptxas_src, dst)
        os.chmod(dst, 0o755)
    os.environ["TRITON_PTXAS_PATH"] = dst
    from triton.backends.nvidia import compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("Triton ptxas patched for Blackwell")
except Exception as e:
    print(f"Triton patch skipped: {e}")

# Patch 2: Pure PyTorch RMSNorm (avoid Triton kernel crashes)
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5, group_size=None,
                     norm_before_gate=True, upcast=None):
    orig_dtype = x.dtype
    x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    x_normed = x_normed.to(orig_dtype)
    out = x_normed * weight
    if bias is not None:
        out = out + bias
    if z is not None:
        z_act = z * torch.sigmoid(z.float()).to(z.dtype)
        out = out * z_act if norm_before_gate else (x_normed * z_act) * weight
    return out

print("RMSNorm pure PyTorch fallback defined")

Triton patch skipped: [Errno 30] Read-only file system: '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/third_party'
RMSNorm pure PyTorch fallback defined


In [4]:
# --- Load model & apply patches ---
import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Model path: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Note: 30B BF16 model needs ~60GB, RTX Pro 6000 has 48GB VRAM.
# Use device_map="auto" to offload some layers to CPU.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16
)

# Apply RMSNorm patch after model loads
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print(f"Model loaded: {model.num_parameters()/1e9:.1f}B params")
print(f"Device map: {set(model.hf_device_map.values()) if hasattr(model, 'hf_device_map') else 'N/A'}")

Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


`torch_dtype` is deprecated! Use `dtype` instead!
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded: 31.6B params
Device map: N/A


## Step 1: Training Data

We use pre-built training data from two datasets:

- `konbu17/nemotron-doc2lora-training-data` — Knowledge QA, encryption short CoT, and solver CoT for 4 puzzle types
- `konbu17/nemotron-bm-et-with-generated-cot` — Generated CoT for Bit Manipulation & Equation Transformation

> **Acknowledgment:** The Bit Manipulation and Equation Transformation CoT data is derived from the excellent dataset by [Ngo Xuan Kien (kienngx)](https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels). We sampled and lightly cleaned it for our pipeline. For the full original dataset with CoT labels for all types, please use [the original](https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels). Thank you Kien for making this available to the community!

### Data Composition

The data is organized into **knowledge injection** data and **task** data:

#### Knowledge Injection Data (Step 2)

| Category | Count | Purpose |
|---|---:|---|
| **Pattern Match QA** | ~1,400 | Teach the model to match partial decryptions like `?oo?` to words (`book`, `door`) |
| **Dictionary Recall** | ~130 | Reinforce memorization of all 77 Wonderland words |
| **Encryption (short CoT)** | ~1,500 | Teach cipher solving WITHOUT listing the full 77-word dictionary in CoT |

**Pattern Match QA example:**
```
Q: In Alice's Wonderland dictionary, which word matches "?oo?"?
A: book, door
```

**Encryption short CoT example** (vs. old approach that listed all 77 words):
```
From the examples, I extract character mappings: u->q, c->u, o->e, ...
Decrypting target: "trb" -> "cat"; "hffk" -> "?oo?" -> matches "book"
```

#### Task SFT Data (Step 3)

| Puzzle Type | Count | CoT Source |
|---|---:|---|
| Bit Manipulation | 800 | Generated CoT (from [kienngx dataset](https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels)) |
| Equation Transformation | 800 | Generated CoT (from [kienngx dataset](https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels)) |
| Gravitational Constant | 400 | Deterministic solver (g=2d/t^2) |
| Unit Conversion | 600 | Deterministic solver (ratio=out/in) |
| Numeral Conversion | 200 | Deterministic solver (Roman numeral) |
| Text Encryption | 600 | Short CoT (same as above, no dictionary listing) |

### The 77-Word Wonderland Dictionary

All Text Encryption problems use words exclusively from this fixed vocabulary:

```
above, alice, ancient, around, beyond, bird, book, bright, castle, cat,
cave, chases, clever, colorful, creates, crystal, curious, dark, discovers,
door, dragon, draws, dreams, explores, follows, forest, found, garden,
golden, hatter, hidden, imagines, in, inside, island, key, king, knight,
library, magical, map, message, mirror, mountain, mouse, mysterious, near,
ocean, palace, potion, princess, puzzle, queen, rabbit, reads, school,
secret, sees, silver, story, strange, student, studies, teacher, the,
through, tower, treasure, turtle, under, valley, village, watches, wise,
wizard, wonderland, writes
```

With only the in-prompt examples, just 38% of problems have complete cipher mappings. With dictionary knowledge, 98% become solvable.

In [5]:
import pandas as pd
import random
from datasets import Dataset as HFDataset

DATA_DIR = "/kaggle/input/datasets/konbu17/nemotron-doc2lora-training-data"
COT_DIR = "/kaggle/input/datasets/konbu17/nemotron-bm-et-with-generated-cot"

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# --- Load knowledge data (Step 2) ---
knowledge_qa = pd.read_csv(os.path.join(DATA_DIR, "knowledge_qa.csv"))
encryption_cot = pd.read_csv(os.path.join(DATA_DIR, "encryption_new_cot.csv"))
print(f"Knowledge QA: {len(knowledge_qa)} | Encryption short CoT: {len(encryption_cot)}")

# --- Load task data (Step 3) ---
SEED = 42

# BM/ET from separate dataset (derived from kienngx)
TASK_SOURCES_COT = [
    (COT_DIR, "train_Bit_Manipulation_with_generated_cot.csv", 800),
    (COT_DIR, "train_Equation_Transformation_with_generated_cot.csv", 800),
]
# Solver CoT from main dataset
TASK_SOURCES_SOLVER = [
    (DATA_DIR, "train_Gravitational_Constant_solver.csv", 400),
    # (DATA_DIR, "train_Gravitational_Constant_solver.csv", 500),
    (DATA_DIR, "train_Unit_Conversion_solver.csv", 600),
    # (DATA_DIR, "train_Unit_Conversion_solver.csv", 700),
    (DATA_DIR, "train_Numeral_Conversion_solver.csv", 200),
    # (DATA_DIR, "train_Numeral_Conversion_solver.csv", 400),    
]

task_dfs = []
for base_dir, fname, n in TASK_SOURCES_COT + TASK_SOURCES_SOLVER:
    df = pd.read_csv(os.path.join(base_dir, fname))
    df_s = df.sample(n=min(n, len(df)), random_state=SEED)
    task_dfs.append(df_s)
    print(f"  {fname}: {len(df)} -> {len(df_s)}")

# Text Encryption uses new short CoT
te_s = encryption_cot.sample(n=min(600, len(encryption_cot)), random_state=SEED)
task_dfs.append(te_s)
print(f"  Text Encryption (short CoT): {len(encryption_cot)} -> {len(te_s)}")

task_df = pd.concat(task_dfs, ignore_index=True).sample(frac=1, random_state=SEED)
print(f"\nTask data total: {len(task_df)}")
print(task_df["type"].value_counts().sort_index())

Knowledge QA: 1553 | Encryption short CoT: 1492
  train_Bit_Manipulation_with_generated_cot.csv: 1506 -> 800
  train_Equation_Transformation_with_generated_cot.csv: 1466 -> 800
  train_Gravitational_Constant_solver.csv: 1516 -> 400
  train_Unit_Conversion_solver.csv: 1513 -> 600
  train_Numeral_Conversion_solver.csv: 1497 -> 200
  Text Encryption (short CoT): 1492 -> 600

Task data total: 3400
type
Bit Manipulation           800
Equation Transformation    800
Gravitational Constant     400
Numeral Conversion         200
Text Encryption            600
Unit Conversion            600
Name: count, dtype: int64


In [6]:
def build_knowledge_dataset(knowledge_qa, encryption_cot):
    """Build Step 2 dataset: knowledge QA + encryption short CoT"""
    records = []
    for _, row in knowledge_qa.iterrows():
        user_content = str(row["question"]) + PROMPT_SUFFIX
        assistant_content = f"{row['cot']}\n</think>\n\\boxed{{{row['answer']}}}"
        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
    for _, row in encryption_cot.iterrows():
        user_content = str(row["prompt"]) + PROMPT_SUFFIX
        assistant_content = f"{row['generated_cot']}\n</think>\n\\boxed{{{row['answer']}}}"
        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
    random.seed(SEED)
    random.shuffle(records)
    return HFDataset.from_list(records)

def build_task_dataset(df):
    """Build Step 3 dataset: all 6 puzzle types in Kaggle format"""
    records = []
    for _, row in df.iterrows():
        user_content = str(row["prompt"]) + PROMPT_SUFFIX
        assistant_content = f"{row['generated_cot']}\n</think>\n\\boxed{{{row['answer']}}}"
        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
    return HFDataset.from_list(records)

knowledge_dataset = build_knowledge_dataset(knowledge_qa, encryption_cot)
task_dataset = build_task_dataset(task_df)
print(f"Knowledge dataset: {len(knowledge_dataset)} | Task dataset: {len(task_dataset)}")

Knowledge dataset: 3045 | Task dataset: 3400


## Step 2: Knowledge Injection SFT (Phase 1)

The first training phase deeply embeds the 77-word dictionary into the MLP layers.

**Key design choices inspired by Doc-to-LoRA:**

- **Higher alpha (64 vs typical 32):** Doc-to-LoRA uses `alpha = r^1.5 * 2` for stronger knowledge injection. We use `alpha=64` (2x rank) as a practical approximation.
- **Higher LR (1e-4):** Faster embedding of factual patterns. Doc-to-LoRA's context distillation uses 1e-4.
- **dropout=0.0:** No dropout during knowledge injection (Doc-to-LoRA default).
- **Target: MLP + Mamba projectors:** `up_proj`/`down_proj` (Transformer FFN) + `in_proj`/`out_proj` (Mamba projectors, validated by [ProDiaL](https://arxiv.org/abs/2411.15224)).

In [7]:
import time, gc
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

# --- LoRA config ---
LORA_RANK = 32
# LORA_ALPHA = 32 # 2x rank, inspired by Doc-to-LoRA's r^1.5*2 scaling
LORA_ALPHA = 32 # 64 # mend→32 # 2x rank, inspired by Doc-to-LoRA's r^1.5*2 scaling

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, # Doc to lora dropout is 0.00 # 0.00 # 0.05
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [8]:
# --- Step 2: Knowledge Injection Training ---
KNOWLEDGE_OUTPUT = "/kaggle/working/knowledge_adapter"

knowledge_args = SFTConfig(
    output_dir=KNOWLEDGE_OUTPUT,
    num_train_epochs=1, # 1 # 1.2 # 1
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8, # 4 # 8
    learning_rate=1e-4,       # High LR for knowledge injection
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=2048, # 4096 # 2048 # 4096 # 2048       # Knowledge QA is short
    logging_steps=50,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=knowledge_args,
    train_dataset=knowledge_dataset,
    processing_class=tokenizer,
)

print(f"Step 2: Knowledge Injection SFT - {len(knowledge_dataset)} samples")
t0 = time.time()
trainer.train()
print(f"Step 2 complete in {(time.time()-t0)/60:.1f} min")

# Save intermediate adapter
model.save_pretrained(KNOWLEDGE_OUTPUT)
tokenizer.save_pretrained(KNOWLEDGE_OUTPUT)
print(f"Knowledge adapter saved to {KNOWLEDGE_OUTPUT}")

# Free trainer memory
del trainer
gc.collect()
torch.cuda.empty_cache()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/3045 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3045 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Step 2: Knowledge Injection SFT - 3045 samples


Step,Training Loss
50,26.779619
100,8.479534
150,7.033271
200,6.946607
250,6.127379
300,6.280181
350,6.172427


Step 2 complete in 205.4 min
Knowledge adapter saved to /kaggle/working/knowledge_adapter


## Step 3: Task SFT (Phase 2)

Continue training from the knowledge-injected adapter on all 6 puzzle types.

- Lower LR (5e-5) to preserve the embedded dictionary knowledge
- Full max_length=7680 for longer generated CoT responses
- All 6 types including Text Encryption with short CoT (the dictionary is now in the weights)

In [9]:
# --- Step 3: Task SFT ---
TASK_OUTPUT = "/kaggle/working/task_adapter"

task_args = SFTConfig(
    output_dir=TASK_OUTPUT,
    num_train_epochs=1.0, # 1 # 1.2 # 1
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8, # 4 # 8
    learning_rate=5e-5, # 5e-5 # 7e-7 # 5e-5     # Lower LR to preserve knowledge
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=2048, # 7680 (My original Param. but OOM?) # 2048 # 4096        # Full length for generated CoT
    logging_steps=50,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=task_args,
    train_dataset=task_dataset,
    processing_class=tokenizer,
)

print(f"Step 3: Task SFT - {len(task_dataset)} samples")
t0 = time.time()
trainer.train()
print(f"Step 3 complete in {(time.time()-t0)/60:.1f} min")

# Save final adapter
model.save_pretrained(TASK_OUTPUT)
tokenizer.save_pretrained(TASK_OUTPUT)
print(f"Task adapter saved to {TASK_OUTPUT}")

del trainer
gc.collect()
torch.cuda.empty_cache()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/3400 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3400 [00:00<?, ? examples/s]

Step 3: Task SFT - 3400 samples


Step,Training Loss
50,16.498074
100,10.261022
150,9.937186
200,9.297102
250,9.345840
300,9.539529
350,9.642228
400,9.576485


Step 3 complete in 257.7 min
Task adapter saved to /kaggle/working/task_adapter


In [10]:
import json, zipfile

OUTPUT_DIR = "/kaggle/working"
ADAPTER_DIR = TASK_OUTPUT

# Fix base_model_name_or_path for Kaggle evaluation
config_path = os.path.join(ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
cfg["inference_mode"] = True
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("Adapter config:")
# print(f"  rank={cfg['r']}, alpha={cfg['lora_alpha']}, dropout={cfg.get('lora_dropout',0.00)}")
print(f"  rank={cfg['r']}, alpha={cfg['lora_alpha']}, dropout={cfg.get('lora_dropout',0.05)}")
print(f"  target={cfg['target_modules']}")
print(f"  base_model={cfg['base_model_name_or_path']}")

# Create submission.zip
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["adapter_config.json", "adapter_model.safetensors"]:
        fpath = os.path.join(ADAPTER_DIR, fname)
        zf.write(fpath, fname)  # store at zip root
        print(f"  Added {fname} ({os.path.getsize(fpath)/1024**2:.1f} MB)")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024**3:.2f} GB")
print("Ready to submit!")

Adapter config:
  rank=32, alpha=32, dropout=0.05
  target=.*\.(in_proj|out_proj|up_proj|down_proj)$
  base_model=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  Added adapter_config.json (0.0 MB)
  Added adapter_model.safetensors (3359.2 MB)

submission.zip: 3.01 GB
Ready to submit!


## Step 4: Create submission.zip

## Inference Demo

Quick sanity check: generate answers for 5 hand-crafted test prompts (one per puzzle type).

In [11]:
import re

# Merge adapter for inference
model = model.merge_and_unload()
model.eval()
gc.collect()
torch.cuda.empty_cache()

DEMO_PROMPTS = [
# Text Encryption - 5 examples, partial mapping
"""In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
ucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley
pqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle
gbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door
bxo sfjpov pqrsfv dfjjfig -> the golden dragon follows
nqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret
Now, decrypt the following text: trb wzrswvog hffk""",

# Text Encryption - 3 examples, fewer mappings available
"""In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
hmxad apdhvdq vid ohexahm apwqvhm -> alice creates the magical crystal
zxuhpl zhvaidq xyqxld txmmhed -> wizard watches inside village
nfddy xohexydq xy ehpldy -> queen imagines in garden
Now, decrypt the following text: bxye aihqdq ahqvmd""",

# Text Encryption - 3 examples, minimal context
"""In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
xkffmh chmkfmn ewhmnf -> hatter creates forest
hkyydf cxknmn chbnfkj -> rabbit chases crystal
shkawg cxknmn dg owugfkdg -> dragon chases in mountain
Now, decrypt the following text: kjdcm pkfcxmn ugsmh pwgsmhjkgs""",
]

DEMO_EXPECTED = [
    "cat imagines book",
    "king chases castle",
    "alice watches under wonderland",
]

print("=" * 60)
print("Inference Demo: Text Encryption (3 prompts)")
print("=" * 60)

for i, (prompt, expected) in enumerate(zip(DEMO_PROMPTS, DEMO_EXPECTED)):
    user_content = prompt + PROMPT_SUFFIX
    messages = [{"role": "user", "content": user_content}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=4096, temperature=0.0, do_sample=False)
    response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

    # Extract boxed answer
    boxed = re.findall(r'\\boxed\{([^}]*)\}', response)
    answer = boxed[-1].strip() if boxed else response.split('\n')[-1].strip()

    print(f"\n{'─'*60}")
    print(f"Demo {i+1} | Expected: {expected}")
    print(f"Model answer: {answer}")
    print(f"Correct: {'YES' if answer.strip().lower() == expected.strip().lower() else 'NO'}")

    # Show thinking (up to 4000 chars)
    think_end = response.find("</think>")
    if think_end > 0:
        thinking = response[:think_end].strip()
        print(f"\nThinking ({len(thinking)} chars):")
        print(thinking[:4000])
        if len(thinking) > 4000:
            print("... [truncated]")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Inference Demo: Text Encryption (3 prompts)

────────────────────────────────────────────────────────────
Demo 1 | Expected: cat imagines book
Model answer: tree writes book
Correct: NO

Thinking (1469 chars):
We need to decrypt the given text: "trb wzrswvog hffk". Let's analyze the given examples to find patterns.

1. "ucoov pwgtfyoqg vorq yrjjoe" -> queen discovers near valley
2. "pqrsfv pqorzg wvgwpo trgbjo" -> dragon dreams inside castle
3. "gbcpovb tqorbog bxo zrswtrj pffq" -> student creates the magical door
4. "nqwvtogg qorpg bxo zegboqwfcg gotqob" -> princess reads the mysterious secret

The target is "trb wzrswvog hffk". Let's look for common words. "trb" is likely "tree". "wzrswvog" is likely "wizard" or "writes". "hffk" is likely "book". So "tree writes book". That makes sense.

So "tree writes book". Let's see if "tree writes book" matches any of the examples. "tree writes book" is not in the examples. "wizard writes book" also doesn't match. "wizard creates door" also does

## Summary

This notebook demonstrated a **Doc-to-LoRA inspired 2-phase knowledge injection** approach:

1. **Phase 1 (Knowledge SFT):** Pattern-matching QA + dictionary recall -> embeds 77-word dictionary into MLP/Mamba projector layers
2. **Phase 2 (Task SFT):** All 6 puzzle types with type-specific CoT -> teaches reasoning patterns on top of injected knowledge

The key novelty is that Text Encryption accuracy **tripled** (22% -> 75%) by moving dictionary knowledge from the CoT (wasting tokens) into the LoRA weights (always available).

If you found this useful, please upvote! Feedback and discussion welcome.